# Step 0: 환경 설정 & 공통 유틸**목적**: 1~3단계 모든 노트북에서 공유하는 경로·스키마·헬퍼 정의.**사용**: 다른 노트북 시작부에서 `from common import *` 처럼 쓸 수 있게 `/content/drive/MyDrive/realestate_reco/common.py`에 저장.**전제**: 설계서 2장(저장소 구조)·3장(통합 키)·5장(스키마) 결정 사항이 이미 합의됨.

## 1. Google Drive 마운트 & 경로 설정설계서 2.1절 디렉토리 레이아웃을 그대로 만든다. 이미 있으면 건너뜀.

In [ ]:
from google.colab import drivedrive.mount('/content/drive')DATA_DIR = '/content/drive/MyDrive/realestate_reco'

In [ ]:
import osfrom pathlib import PathBASE = Path(DATA_DIR)# 설계서 2.1절 구조 그대로DIRS = {    'raw_trade':       BASE / 'raw' / 'rtms_trade',    'raw_rent':        BASE / 'raw' / 'rtms_rent',    'raw_apt_info':    BASE / 'raw' / 'apt_info',    'raw_pop':         BASE / 'raw' / 'pop',    'raw_school':      BASE / 'raw' / 'school',    'raw_shop':        BASE / 'raw' / 'shop',    'staged_trade':    BASE / 'staged' / 'trade',    'staged_rent':     BASE / 'staged' / 'rent',    'staged_apt_info': BASE / 'staged' / 'apt_info_snapshots',    'staged_pop':      BASE / 'staged' / 'pop_snapshots',    'staged_school':   BASE / 'staged' / 'school_snapshots',    'staged_shop':     BASE / 'staged' / 'shop_snapshots',    'master':          BASE / 'master',    'exports_daily':   BASE / 'exports' / 'daily',    'exports_weekly':  BASE / 'exports' / 'weekly',    'exports_monthly': BASE / 'exports' / 'monthly',    'logs':            BASE / 'logs',    'config':          BASE / 'config',}for name, path in DIRS.items():    path.mkdir(parents=True, exist_ok=True)    print(f'  {name:18s} → {path}')print('\n디렉토리 구조 준비 완료')

## 2. 환경변수·시크릿 관리**원칙**: API 키는 코드에 박지 않는다. `config/secrets.json` 또는 Colab Secrets 사용.```json// config/secrets.json (이 파일은 절대 git에 커밋 금지){  "data_go_kr_service_key": "본인키...",  "kapt_service_key": "본인키...",  "slack_webhook": "https://hooks.slack.com/...",  "MOCK_MODE": false}````MOCK_MODE: true`로 두면 API 키 없이 가상 데이터로 파이프라인 동작 확인 가능.

In [ ]:
import jsonSECRETS_PATH = DIRS['config'] / 'secrets.json'# 첫 실행 시 템플릿 자동 생성if not SECRETS_PATH.exists():    template = {        "data_go_kr_service_key": "PUT_YOUR_KEY_HERE",        "kapt_service_key": "PUT_YOUR_KEY_HERE",        "slack_webhook": "",        "MOCK_MODE": True,    }    SECRETS_PATH.write_text(json.dumps(template, ensure_ascii=False, indent=2), encoding='utf-8')    print(f'템플릿 생성: {SECRETS_PATH}')    print('이 파일을 열어서 API 키 입력 후 다시 실행하세요. 일단 MOCK_MODE=True로 진행 가능.')SECRETS = json.loads(SECRETS_PATH.read_text(encoding='utf-8'))MOCK_MODE = SECRETS.get('MOCK_MODE', True)print(f'MOCK_MODE = {MOCK_MODE}')

## 3. 수도권 시군구 코드설계서 1.1절 — RTMS API는 시군구 5자리 코드(LAWD_CD)로 호출.서울 25 + 경기 31 + 인천 10 = 66개.

In [ ]:
# 수도권 시군구 코드 (행안부 행정표준코드 앞 5자리)SIGUNGU_CODES = {    # 서울 25    '11110': '종로구',  '11140': '중구',    '11170': '용산구',  '11200': '성동구',    '11215': '광진구',  '11230': '동대문구','11260': '중랑구',  '11290': '성북구',    '11305': '강북구',  '11320': '도봉구',  '11350': '노원구',  '11380': '은평구',    '11410': '서대문구','11440': '마포구',  '11470': '양천구',  '11500': '강서구',    '11530': '구로구',  '11545': '금천구',  '11560': '영등포구','11590': '동작구',    '11620': '관악구',  '11650': '서초구',  '11680': '강남구',  '11710': '송파구',    '11740': '강동구',    # 인천 10    '28110': '중구(인천)','28140': '동구(인천)','28177': '미추홀구','28185': '연수구',    '28200': '남동구',  '28237': '부평구',  '28245': '계양구',  '28260': '서구(인천)',    '28710': '강화군',  '28720': '옹진군',    # 경기 31    '41111': '수원시 장안구','41113': '수원시 권선구','41115': '수원시 팔달구','41117': '수원시 영통구',    '41131': '성남시 수정구','41133': '성남시 중원구','41135': '성남시 분당구',    '41150': '의정부시',  '41171': '안양시 만안구','41173': '안양시 동안구',    '41190': '부천시',    '41210': '광명시',    '41220': '평택시',    '41250': '동두천시',    '41271': '안산시 상록구','41273': '안산시 단원구',    '41281': '고양시 덕양구','41285': '고양시 일산동구','41287': '고양시 일산서구',    '41290': '과천시',    '41310': '구리시',    '41360': '남양주시',    '41370': '오산시',    '41390': '시흥시',    '41410': '군포시',    '41430': '의왕시',    '41450': '하남시',    '41461': '용인시 처인구','41463': '용인시 기흥구','41465': '용인시 수지구',    '41480': '파주시',    '41500': '이천시',    '41550': '안성시',    '41570': '김포시',    '41590': '화성시',    '41610': '광주시',    '41630': '양주시',    '41650': '포천시',    '41670': '여주시',    '41800': '연천군',    '41820': '가평군',    '41830': '양평군',}print(f'수도권 시군구 총 {len(SIGUNGU_CODES)}개')

## 4. 평형 버킷 함수설계서 3.4절 — 전용면적(㎡) → 평형 버킷.이 함수는 모든 단계에서 동일하게 쓰여야 한다.

In [ ]:
def to_pyeong_bucket(area_m2: float) -> str:    """전용면적(㎡)을 9단계 평형 버킷으로 변환.    설계서 3.4절 표에 따른 분류.    경계값 처리: 미만(<) 기준이 아닌 이하(≤) 기준 ─ 84.99㎡는 30P 끝자락이 아니라 30P.    """    if area_m2 < 33:    return '10P'    if area_m2 < 50:    return '15P'    if area_m2 < 66:    return '20P'    if area_m2 < 83:    return '25P'    if area_m2 < 99:    return '30P'    if area_m2 < 116:   return '35P'    if area_m2 < 132:   return '40P'    if area_m2 < 165:   return '50P'    return '60P+'# 테스트test_cases = [(32.0, '10P'), (84.99, '30P'), (85.0, '30P'), (99.0, '35P'), (200.0, '60P+')]for m2, expected in test_cases:    got = to_pyeong_bucket(m2)    mark = '✓' if got == expected else '✗'    print(f'  {mark} {m2:6.2f}㎡ → {got:5s} (expected {expected})')

## 5. 단지명 정규화 함수설계서 3.3절 — 같은 단지가 여러 표기로 등장하는 문제 해결.

In [ ]:
import redef normalize_apt_name(name: str) -> str:    """단지명 정규화. 설계서 3.3절 규칙 그대로.    1. 괄호 내용 제거    2. 공백 모두 제거    3. 'APT'/'apt' → '아파트'    4. 한글 숫자 → 아라비아 숫자    """    if name is None:        return ''    s = str(name).strip()    # 1. 괄호 내용 제거 (한글·영문·숫자 괄호 모두)    s = re.sub(r'\([^)]*\)', '', s)    s = re.sub(r'\[[^\]]*\]', '', s)    # 2. 공백 제거    s = re.sub(r'\s+', '', s)    # 3. APT 표기 통일    s = re.sub(r'(?i)apt$', '아파트', s)    s = s.replace('아파트아파트', '아파트')    # 4. 한글 숫자 → 아라비아 숫자 (1차/일차, 2차/이차 …)    han2num = {'일': '1', '이': '2', '삼': '3', '사': '4', '오': '5',               '육': '6', '칠': '7', '팔': '8', '구': '9', '십': '10'}    for han, num in han2num.items():        s = s.replace(han + '차', num + '차')        s = s.replace(han + '단지', num + '단지')    return s# 테스트tests = [    ('대치동부센트레빌', '대치동부센트레빌'),    ('대치 동부센트레빌(101동)', '대치동부센트레빌'),    ('래미안 일차', '래미안1차'),    ('힐스테이트APT', '힐스테이트아파트'),    ('자이[A동]', '자이'),]for inp, expected in tests:    got = normalize_apt_name(inp)    mark = '✓' if got == expected else '✗'    print(f'  {mark} {inp!r:30s} → {got!r}')

## 6. 통합 단지 ID (apt_id) 생성 함수설계서 3.2절 — SHA256 해시 16자리.**같은 입력에서 항상 같은 출력**이 보장돼야 한다 (재현성).

In [ ]:
import hashlibdef make_apt_id(legal_dong_code: str,                apt_name: str,                road_address: str = '') -> str:    """통합 단지 ID 생성. 설계서 3.2절.    - legal_dong_code: 법정동 10자리 (예: '1168010600')    - apt_name: 정규화 전 원본 단지명 (내부에서 정규화 호출)    - road_address: 도로명주소 (있으면 앞 30자만 사용, 없으면 빈 문자열)    같은 단지면 같은 apt_id, 다른 단지면 다른 apt_id가 나와야 한다.    """    norm = normalize_apt_name(apt_name)    addr = (road_address or '')[:30]    raw = f'{legal_dong_code}|{norm}|{addr}'    h = hashlib.sha256(raw.encode('utf-8')).hexdigest()    return h[:16]# 테스트print(make_apt_id('1168010600', '대치동부센트레빌', '서울특별시 강남구 도곡로'))print(make_apt_id('1168010600', '대치 동부센트레빌(101동)', '서울특별시 강남구 도곡로'))  # 위와 같아야 함print(make_apt_id('1168010600', '대치동부센트레빌2차', '서울특별시 강남구 도곡로'))  # 달라야 함

## 7. 공통 모듈 저장위 헬퍼들을 `common.py`로 묶어 다른 노트북에서 import.

In [ ]:
COMMON_PY = '''"""공통 유틸 - 1~3단계 모든 노트북이 import.이 파일은 Step 0 노트북에서 자동 생성된다."""import reimport hashlibimport jsonfrom pathlib import PathDATA_DIR = '/content/drive/MyDrive/realestate_reco'BASE = Path(DATA_DIR)DIRS = {{    'raw_trade':       BASE / 'raw' / 'rtms_trade',    'raw_rent':        BASE / 'raw' / 'rtms_rent',    'raw_apt_info':    BASE / 'raw' / 'apt_info',    'staged_trade':    BASE / 'staged' / 'trade',    'staged_rent':     BASE / 'staged' / 'rent',    'staged_apt_info': BASE / 'staged' / 'apt_info_snapshots',    'master':          BASE / 'master',    'exports_daily':   BASE / 'exports' / 'daily',    'logs':            BASE / 'logs',    'config':          BASE / 'config',}}SIGUNGU_CODES = {sigungu_dump}def load_secrets():    p = BASE / 'config' / 'secrets.json'    return json.loads(p.read_text(encoding='utf-8'))def to_pyeong_bucket(area_m2: float) -> str:    if area_m2 < 33:    return '10P'    if area_m2 < 50:    return '15P'    if area_m2 < 66:    return '20P'    if area_m2 < 83:    return '25P'    if area_m2 < 99:    return '30P'    if area_m2 < 116:   return '35P'    if area_m2 < 132:   return '40P'    if area_m2 < 165:   return '50P'    return '60P+'def normalize_apt_name(name) -> str:    if name is None:        return ''    s = str(name).strip()    s = re.sub(r'\\([^)]*\\)', '', s)    s = re.sub(r'\\[[^\\]]*\\]', '', s)    s = re.sub(r'\\s+', '', s)    s = re.sub(r'(?i)apt$', '아파트', s)    s = s.replace('아파트아파트', '아파트')    han2num = {{'일':'1','이':'2','삼':'3','사':'4','오':'5','육':'6','칠':'7','팔':'8','구':'9','십':'10'}}    for han, num in han2num.items():        s = s.replace(han+'차', num+'차')        s = s.replace(han+'단지', num+'단지')    return sdef make_apt_id(legal_dong_code, apt_name, road_address='') -> str:    norm = normalize_apt_name(apt_name)    addr = (road_address or '')[:30]    raw = f'{{legal_dong_code}}|{{norm}}|{{addr}}'    return hashlib.sha256(raw.encode('utf-8')).hexdigest()[:16]'''.format(sigungu_dump=repr(SIGUNGU_CODES))(BASE / 'common.py').write_text(COMMON_PY, encoding='utf-8')print(f'저장: {BASE / "common.py"}')

---## 다음 단계- **Step 1a**: RTMS 매매 실거래가 수집 (월별·시군구별 호출)- **Step 1b**: K-apt 단지정보 수집 (월 스냅샷)- **Step 2**: staged 정제 + apt_id_map 빌드- **Step 3**: master 통합 데이터셋 빌드